In [ ]:
# Import libraries and configuration
from pathlib import Path
import math
import json
import requests
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from shapely.geometry import shape
from rasterio.errors import RasterioIOError

# --------------------------------------------------
# Project paths
# --------------------------------------------------

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "spatial_metadata" else Path.cwd()

DATASET_DIR = PROJECT_DIR / "dataset"
DATA_DIR = DATASET_DIR / "data"

REFERENCE_DIR = DATA_DIR / "reference"
ELEVATION_DIR = DATA_DIR / "elevation"
METADATA_DIR = DATASET_DIR / "metadata"

for directory in [REFERENCE_DIR, ELEVATION_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

GRID_CSV = METADATA_DIR / "grid_points.csv"

BIG_GPKG = REFERENCE_DIR / "big_admin_kabupaten_kota.gpkg"
ELEVATION_CSV = ELEVATION_DIR / "elevation_copernicus.csv"

OUTPUT_CSV = METADATA_DIR / "grid_points_enriched.csv"

# --------------------------------------------------
# Spatial configuration
# --------------------------------------------------

CRS_GEOGRAPHIC = "EPSG:4326"
CRS_METRIC = "EPSG:6933"

MAX_NEAREST_DISTANCE_M = 2000

BIG_LAYER = "admin_kabupaten_kota"

BIG_FEATURE_URL = (
    "https://geoservices.big.go.id/rbi/rest/services/"
    "Hosted/Wilayah_Administrasi_Kabupaten__Kota/"
    "FeatureServer/0"
)

print("Project directory:", PROJECT_DIR)
print("Grid input:", GRID_CSV)
print("BIG reference:", BIG_GPKG)
print("Elevation cache:", ELEVATION_CSV)
print("Output:", OUTPUT_CSV)

In [ ]:
# Check the input grid
if not GRID_CSV.exists():
    raise FileNotFoundError(
        f"Grid file not found: {GRID_CSV}. "
        "Download the Mendeley dataset and place it in dataset/metadata/."
    )

grid = pd.read_csv(GRID_CSV)

required_columns = {"grid_id", "latitude", "longitude"}

missing_columns = required_columns - set(grid.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

if grid["grid_id"].duplicated().any():
    raise ValueError("Duplicate grid_id values were detected.")

if grid[["latitude", "longitude"]].isna().any().any():
    raise ValueError("Missing grid coordinates were detected.")

print("Number of grid points:", len(grid))
display(grid.head())

In [ ]:
# Download BIG administrative boundaries

def download_big_boundaries(output_path):
    """
    Download BIG administrative boundaries through the ArcGIS REST API
    and save the resulting polygons as a GeoPackage.
    """

    query_url = f"{BIG_FEATURE_URL}/query"

    all_features = []
    offset = 0
    batch_size = 1000

    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "true",
            "outSR": 4326,
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": batch_size,
        }

        response = requests.get(
            query_url,
            params=params,
            timeout=180
        )
        response.raise_for_status()

        result = response.json()

        if "features" not in result:
            raise RuntimeError(
                f"BIG API response did not contain features: {result}"
            )

        features = result["features"]

        if not features:
            break

        all_features.extend(features)
        offset += len(features)

        print("Downloaded features:", len(all_features))

        if len(features) < batch_size:
            break

    if not all_features:
        raise RuntimeError("No administrative boundaries were downloaded.")

    records = []

    for feature in all_features:
        records.append({
            "geometry": shape(feature["geometry"]),
            **feature["properties"]
        })

    boundaries = gpd.GeoDataFrame(
        records,
        geometry="geometry",
        crs=CRS_GEOGRAPHIC
    )

    boundaries = boundaries[
        boundaries.geometry.notna()
        & ~boundaries.geometry.is_empty
    ].copy()

    if "wadmpr" not in boundaries.columns:
        raise ValueError("The BIG data do not contain the wadmpr field.")

    boundaries = boundaries[
        ["wadmpr", "geometry"]
    ].copy()

    boundaries.to_file(
        output_path,
        layer=BIG_LAYER,
        driver="GPKG"
    )

    print("Saved BIG boundaries:", output_path)
    print("Number of administrative polygons:", len(boundaries))

    return boundaries


if BIG_GPKG.exists():
    print("Existing BIG reference found. Reading local file.")

    boundaries = gpd.read_file(
        BIG_GPKG,
        layer=BIG_LAYER
    )

else:
    print("BIG reference not found. Downloading from BIG.")

    boundaries = download_big_boundaries(BIG_GPKG)

boundaries = boundaries.to_crs(CRS_GEOGRAPHIC)

display(boundaries.head())

In [ ]:
# Assign province to each grid point

points = gpd.GeoDataFrame(
    grid.copy(),
    geometry=gpd.points_from_xy(
        grid["longitude"],
        grid["latitude"]
    ),
    crs=CRS_GEOGRAPHIC
)

# Direct spatial matching
direct_join = gpd.sjoin(
    points,
    boundaries[["wadmpr", "geometry"]],
    how="left",
    predicate="intersects"
)

province_records = []

for grid_id, group in direct_join.groupby("grid_id"):

    provinces = group["wadmpr"].dropna().unique()

    if len(provinces) == 1:
        province_records.append({
            "grid_id": grid_id,
            "province": provinces[0],
            "province_assignment_method": "point_in_polygon",
            "distance_m": np.nan
        })

    elif len(provinces) == 0:
        province_records.append({
            "grid_id": grid_id,
            "province": np.nan,
            "province_assignment_method": np.nan,
            "distance_m": np.nan
        })

    else:
        province_records.append({
            "grid_id": grid_id,
            "province": np.nan,
            "province_assignment_method": "ambiguous_intersection",
            "distance_m": np.nan
        })

province_result = pd.DataFrame(province_records)

# Identify points requiring nearest-polygon matching
unresolved_ids = province_result.loc[
    province_result["province"].isna(),
    "grid_id"
]

unresolved_points = points[
    points["grid_id"].isin(unresolved_ids)
].copy()

if not unresolved_points.empty:

    points_metric = unresolved_points.to_crs(CRS_METRIC)

    boundaries_metric = boundaries.to_crs(CRS_METRIC)

    nearest_join = gpd.sjoin_nearest(
        points_metric,
        boundaries_metric[["wadmpr", "geometry"]],
        how="left",
        distance_col="distance_m"
    )

    nearest_records = []

    for grid_id, group in nearest_join.groupby("grid_id"):

        minimum_distance = group["distance_m"].min()

        nearest_candidates = group[
            group["distance_m"] <= minimum_distance + 0.01
        ]["wadmpr"].dropna().unique()

        if (
            len(nearest_candidates) == 1
            and minimum_distance <= MAX_NEAREST_DISTANCE_M
        ):
            nearest_records.append({
                "grid_id": grid_id,
                "province": nearest_candidates[0],
                "province_assignment_method": "nearest_polygon_within_2km",
                "distance_m": minimum_distance
            })

        else:
            nearest_records.append({
                "grid_id": grid_id,
                "province": np.nan,
                "province_assignment_method": "unresolved",
                "distance_m": minimum_distance
            })

    nearest_result = pd.DataFrame(nearest_records)

    province_result = province_result.set_index("grid_id")

    nearest_result = nearest_result.set_index("grid_id")

    province_result.update(nearest_result)

    province_result = province_result.reset_index()

if province_result["province"].isna().any():
    raise ValueError(
        "Some grid points could not be assigned to a province. "
        "Inspect the unresolved records before proceeding."
    )

print("Province assignment:")
print(province_result["province_assignment_method"].value_counts())

print("Missing province:", province_result["province"].isna().sum())

display(province_result.head())

In [ ]:
# Assign island groups

component_to_island_group = {
    # Kalimantan
    4899: "Kalimantan",
    5417: "Kalimantan",
    6480: "Kalimantan",
    6674: "Kalimantan",
    6646: "Kalimantan",

    # Sumatra
    301: "Sumatra",
    3646: "Sumatra",
    5050: "Sumatra",
    2921: "Sumatra",
    1132: "Sumatra",
    1138: "Sumatra",
    1190: "Sumatra",
    6: "Sumatra",
    544: "Sumatra",
    4427: "Sumatra",
    2121: "Sumatra",
    452: "Sumatra",

    # Papua
    15721: "Papua",
    18861: "Papua",
    12336: "Papua",
    15250: "Papua",
    14306: "Papua",
    18164: "Papua",
    18321: "Papua",
    19170: "Papua",

    # Sulawesi
    6943: "Sulawesi",
    7682: "Sulawesi",
    9136: "Sulawesi",
    8430: "Sulawesi",
    8398: "Sulawesi",

    # Java
    3451: "Java",
    5724: "Java",

    # Bali and Nusa Tenggara
    7357: "Bali and Nusa Tenggara",
    6697: "Bali and Nusa Tenggara",
    9673: "Bali and Nusa Tenggara",
    6022: "Bali and Nusa Tenggara",
    5462: "Bali and Nusa Tenggara",
    6061: "Bali and Nusa Tenggara",
    9677: "Bali and Nusa Tenggara",
    7348: "Bali and Nusa Tenggara",
    8864: "Bali and Nusa Tenggara",

    # Maluku
    11557: "Maluku",
    11893: "Maluku",
    10994: "Maluku",
    11483: "Maluku",
    11541: "Maluku",
    10592: "Maluku",
    12270: "Maluku",
    15442: "Maluku",
    11486: "Maluku",
}

# Combine administrative geometries into island components
union_geometry = boundaries.geometry.union_all()

if union_geometry.geom_type == "Polygon":
    island_geometries = [union_geometry]

elif union_geometry.geom_type == "MultiPolygon":
    island_geometries = list(union_geometry.geoms)

else:
    raise ValueError(
        f"Unexpected geometry type: {union_geometry.geom_type}"
    )

island_components = gpd.GeoDataFrame(
    {
        "component_id": range(len(island_geometries)),
        "geometry": island_geometries
    },
    geometry="geometry",
    crs=CRS_GEOGRAPHIC
)

island_components["island_group"] = (
    island_components["component_id"].map(
        component_to_island_group
    )
)

unmapped_components = island_components[
    island_components["island_group"].isna()
]

if not unmapped_components.empty:
    warnings.warn(
        f"{len(unmapped_components)} island components "
        "are not included in the manual mapping."
    )

# Spatial matching
island_join = gpd.sjoin(
    points,
    island_components[
        ["component_id", "island_group", "geometry"]
    ],
    how="left",
    predicate="within"
)

island_records = []

for grid_id, group in island_join.groupby("grid_id"):

    assigned_groups = group["island_group_right"].dropna().unique()

    if len(assigned_groups) == 1:
        island_group = assigned_groups[0]

    elif len(assigned_groups) == 0:
        island_group = group["island_group_left"].iloc[0]

    else:
        island_group = None

    island_records.append({
        "grid_id": grid_id,
        "island_group": island_group
    })

island_result = pd.DataFrame(island_records)

# Nearest-component matching for unmatched points
unmatched_ids = island_result.loc[
    island_result["island_group"].isna(),
    "grid_id"
]

if len(unmatched_ids) > 0:

    unmatched_points = points[
        points["grid_id"].isin(unmatched_ids)
    ].to_crs(CRS_METRIC)

    components_metric = island_components[
        island_components["island_group"].notna()
    ].to_crs(CRS_METRIC)

    nearest_islands = gpd.sjoin_nearest(
        unmatched_points,
        components_metric[
            ["component_id", "island_group", "geometry"]
        ],
        how="left",
        distance_col="distance_m"
    )

    nearest_records = []

    for grid_id, group in nearest_islands.groupby("grid_id"):

        minimum_distance = group["distance_m"].min()

        candidates = group[
            group["distance_m"] <= minimum_distance + 0.01
        ]["island_group"].dropna().unique()

        if len(candidates) == 1:
            nearest_records.append({
                "grid_id": grid_id,
                "island_group": candidates[0]
            })

        else:
            nearest_records.append({
                "grid_id": grid_id,
                "island_group": np.nan
            })

    nearest_result = pd.DataFrame(nearest_records)

    island_result = island_result.set_index("grid_id")
    nearest_result = nearest_result.set_index("grid_id")

    island_result.update(nearest_result)

    island_result = island_result.reset_index()

if island_result["island_group"].isna().any():
    raise ValueError(
        "Some grid points could not be assigned to an island group."
    )

print("Island group distribution:")
print(island_result["island_group"].value_counts())

display(island_result.head())

In [ ]:
# Extract elevation from Copernicus DEM GLO-30

COPERNICUS_BASE_URL = (
    "https://copernicus-dem-30m.s3.eu-central-1.amazonaws.com"
)

def get_copernicus_tile_name(latitude, longitude):

    lat_degree = math.floor(latitude)
    lon_degree = math.floor(longitude)

    lat_direction = "N" if lat_degree >= 0 else "S"
    lon_direction = "E" if lon_degree >= 0 else "W"

    tile_name = (
        f"Copernicus_DSM_COG_10_"
        f"{lat_direction}{abs(lat_degree):02d}_00_"
        f"{lon_direction}{abs(lon_degree):03d}_00_DEM"
    )

    return tile_name



def get_elevation_for_tile(tile_name, coordinates):

    tile_url = (
        f"{COPERNICUS_BASE_URL}/"
        f"{tile_name}/{tile_name}.tif"
    )

    vsi_url = f"/vsicurl/{tile_url}"

    with rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif"
    ):

        with rasterio.open(vsi_url) as src:

            values = []

            for longitude, latitude in coordinates:

                sample = next(
                    src.sample(
                        [(longitude, latitude)],
                        masked=True
                    )
                )

                if np.ma.is_masked(sample[0]):
                    elevation = np.nan
                else:
                    elevation = float(sample[0])

                values.append(elevation)

    return values

def extract_elevation(grid):

    if ELEVATION_CSV.exists():

        cached = pd.read_csv(ELEVATION_CSV)

        if (
            len(cached) == len(grid)
            and cached["grid_id"].is_unique
            and set(cached["grid_id"]) == set(grid["grid_id"])
            and cached["elevation_m"].notna().all()
        ):

            print("Using existing elevation cache.")

            return cached

        print("Existing elevation cache is incomplete. Re-extracting.")

    elevation_data = grid[
        ["grid_id", "latitude", "longitude"]
    ].copy()

    elevation_data["tile_name"] = elevation_data.apply(
        lambda row: get_copernicus_tile_name(
            row["latitude"],
            row["longitude"]
        ),
        axis=1
    )

    elevation_data["elevation_m"] = np.nan

    for tile_name, group in elevation_data.groupby("tile_name"):

        print("Processing:", tile_name)

        coordinates = list(
            zip(group["longitude"], group["latitude"])
        )

        values = get_elevation_for_tile(
            tile_name,
            coordinates
        )

        elevation_data.loc[
            group.index,
            "elevation_m"
        ] = values

    elevation_result = elevation_data[
        ["grid_id", "elevation_m"]
    ].copy()

    if elevation_result["elevation_m"].isna().any():
        missing = elevation_result[
            elevation_result["elevation_m"].isna()
        ]

        raise ValueError(
            f"Elevation is missing for {len(missing)} grid points."
        )

    elevation_result.to_csv(
        ELEVATION_CSV,
        index=False
    )

    print("Elevation cache saved:", ELEVATION_CSV)

    return elevation_result


elevation_result = extract_elevation(grid)

print(elevation_result["elevation_m"].describe())

display(elevation_result.head())

In [ ]:
zero_points = elevation_result[
    elevation_result["elevation_m"] == 0
].merge(
    grid[["grid_id", "latitude", "longitude"]],
    on="grid_id",
    how="left"
)

zero_points["tile_name"] = zero_points.apply(
    lambda row: get_copernicus_tile_name(
        row["latitude"],
        row["longitude"]
    ),
    axis=1
)

for tile_name, group in zero_points.groupby("tile_name"):

    tile_url = (
        f"{COPERNICUS_BASE_URL}/"
        f"{tile_name}/{tile_name}.tif"
    )

    with rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif"
    ):
        with rasterio.open(f"/vsicurl/{tile_url}") as src:

            print("\nTile:", tile_name)
            print("Raster NoData:", src.nodata)
            print("Raster dtype:", src.dtypes)
            print("Mask flags:", src.mask_flag_enums)

            for _, row in group.head(3).iterrows():

                raw_sample = next(
                    src.sample([(
                        row["longitude"],
                        row["latitude"]
                    )])
                )

                masked_sample = next(
                    src.sample(
                        [(row["longitude"], row["latitude"])],
                        masked=True
                    )
                )

                print(
                    row["grid_id"],
                    "raw:", raw_sample[0],
                    "masked:", masked_sample[0],
                    "is_masked:", np.ma.is_masked(masked_sample[0])
                )

In [ ]:
# Integrate all spatial attributes

# Merge province information
final_grid = grid.merge(
    province_result,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

# Merge island group information
final_grid = final_grid.merge(
    island_result,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

# Merge elevation information
final_grid = final_grid.merge(
    elevation_result,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

# Select final columns
final_columns = [
    "grid_id",
    "latitude",
    "longitude",
    "province",
    "province_assignment_method",
    "distance_m",
    "island_group",
    "elevation_m"
]

final_grid = final_grid[final_columns]

display(final_grid.head())

In [ ]:
# Validate the final metadata

# Basic checks
assert len(final_grid) == len(grid), "Grid count changed."

assert final_grid["grid_id"].is_unique, (
    "Duplicate grid IDs detected."
)

required_attributes = [
    "province",
    "island_group",
    "elevation_m"
]

missing_summary = final_grid[
    required_attributes
].isna().sum()

print("Missing attribute values:")
print(missing_summary)

assert missing_summary.sum() == 0, (
    "Missing spatial attributes were detected."
)

# Province assignment methods
print("\nProvince assignment methods:")
print(
    final_grid["province_assignment_method"].value_counts()
)

# Island group distribution
print("\nIsland group distribution:")
print(final_grid["island_group"].value_counts())

# Province distribution
print("\nProvince distribution:")
print(final_grid["province"].value_counts())

# Elevation statistics
print("\nElevation statistics:")
print(final_grid["elevation_m"].describe())

print("\nValidation completed successfully.")

In [ ]:
# Export the enriched grid metadata

final_grid.to_csv(
    OUTPUT_CSV,
    index=False
)

print("Final metadata saved successfully.")
print("Output file:", OUTPUT_CSV)
print("Number of grid points:", len(final_grid))
print("Number of columns:", len(final_grid.columns))

In [ ]:
# Final output preview

final_check = pd.read_csv(OUTPUT_CSV)

print("Final file:", OUTPUT_CSV)
print("Rows:", len(final_check))
print("Columns:", list(final_check.columns))

display(final_check.head(10))